# 21 · Infrastructure & MLOps — The Machine That Runs the Machine

**Part 2 · Machine Learning Systems** · *Prerequisites: 14–20* · *Book mapping: DMLS ch. 10–11*

Notebooks 13–20 built every stage of the ML loop by hand. Production means the loop runs
**without you**: scheduled, retried, gated, versioned, and reversible at 3 a.m. That is
infrastructure, and its layers stack like this: storage & compute (the cloud's job) →
**resource management & orchestration** (today's first build) → experiment/dev tooling
(nb 17's tracker) → the **ML platform**: feature store (nb 16), **model registry**
(today's second build), monitoring (nb 19), serving (nb 18).

The orchestration lineage in one line: **cron** runs commands at times (no dependencies,
no retries — the 4 a.m. training job starts whether or not the 2 a.m. ingest finished);
**schedulers** add queues and resources; **orchestrators** (Airflow, Prefect, Dagster)
run **DAGs** — dependency graphs with retries, backfills, and observability. We build the
essence of one, then wire the whole course's loop through it.

**Learning objectives.**

1. Build a mini **workflow orchestrator**: DAG validation (cycles fail loudly),
   topological execution, retries, failure isolation, and cached backfills.
2. Build a **model registry** with lifecycle stages and *enforced* promotion gates.
3. Compose the full continual-learning pipeline — ingest → validate → features → train →
   evaluate → promote — and watch the gate stop a bad model automatically.
4. Reason about build-vs-buy and team shape (the human infrastructure).

In [ ]:
import hashlib
import json

import numpy as np
from sklearn.linear_model import LogisticRegression

from sysdes import viz

viz.use_style()
rng = np.random.default_rng(seed=21)

## 1. Build: a mini orchestrator

The contract: tasks declare **dependencies**; the engine guarantees order, retries
transient failures, isolates permanent ones (downstream blocked, independent branches
continue), and — because task outputs are keyed by their run context — never re-does
finished work (which is what makes **backfills** cheap; nb 10's determinism rule is
load-bearing again).

In [ ]:
class CycleError(Exception):
    pass

class Pipeline:
    """A DAG of named tasks. States per run: SUCCESS, CACHED, FAILED,
    UPSTREAM_FAILED. Retries happen inside a run; caching spans runs."""

    def __init__(self, name: str):
        self.name = name
        self.tasks: dict[str, dict] = {}         # name -> {fn, deps, retries}
        self.completed: dict[tuple, object] = {} # (task, run_key) -> output
        self.history: list[dict] = []

    def task(self, name: str, deps: tuple = (), retries: int = 0):
        def decorator(fn):
            self.tasks[name] = {"fn": fn, "deps": tuple(deps), "retries": retries}
            return fn
        return decorator

    # ------------------------------------------------------------ validation
    def _toposort(self) -> list[str]:
        """Kahn's algorithm; leftovers mean a cycle - refuse loudly, because a
        cyclic 'schedule' would either deadlock or run forever."""
        indegree = {t: len(spec["deps"]) for t, spec in self.tasks.items()}
        for t, spec in self.tasks.items():
            for d in spec["deps"]:
                if d not in self.tasks:
                    raise KeyError(f"task {t!r} depends on unknown task {d!r}")
        order = [t for t, deg in indegree.items() if deg == 0]
        frontier = list(order)
        while frontier:
            u = frontier.pop()
            for t, spec in self.tasks.items():
                if u in spec["deps"]:
                    indegree[t] -= 1
                    if indegree[t] == 0:
                        order.append(t)
                        frontier.append(t)
        if len(order) != len(self.tasks):
            stuck = sorted(set(self.tasks) - set(order))
            raise CycleError(f"dependency cycle involving {stuck}")
        return order

    # ------------------------------------------------------------- execution
    def run(self, run_key: str, ctx: dict | None = None) -> dict[str, str]:
        ctx = ctx or {}
        states: dict[str, str] = {}
        outputs: dict[str, object] = {}
        for name in self._toposort():
            spec = self.tasks[name]
            if any(states.get(d) in ("FAILED", "UPSTREAM_FAILED")
                   for d in spec["deps"]):
                states[name] = "UPSTREAM_FAILED"     # blocked, not attempted
                continue
            if (name, run_key) in self.completed:    # done in a previous run
                states[name] = "CACHED"
                outputs[name] = self.completed[(name, run_key)]
                continue
            dep_outputs = {d: outputs[d] for d in spec["deps"]}
            for attempt in range(spec["retries"] + 1):
                try:
                    result = spec["fn"](ctx, dep_outputs)
                    self.completed[(name, run_key)] = result
                    outputs[name] = result
                    states[name] = "SUCCESS"
                    break
                except Exception as e:
                    if attempt == spec["retries"]:
                        states[name] = "FAILED"
                        print(f"  [{name}] FAILED after {attempt + 1} attempt(s): {e}")
                    else:
                        print(f"  [{name}] attempt {attempt + 1} failed ({e}) - retrying")
        self.history.append({"run_key": run_key, "states": dict(states)})
        return states

# --- Cycle detection, demonstrated:
cyclic = Pipeline("nonsense")
cyclic.task("a", deps=("b",))(lambda ctx, dep: None)
cyclic.task("b", deps=("a",))(lambda ctx, dep: None)
try:
    cyclic.run("today")
    raise AssertionError("must refuse cycles")
except CycleError as e:
    print(f"cycle refused: {e}")

In [ ]:
# --- Retries + failure isolation, demonstrated on a diamond DAG:
#       ingest -> features -> train         (flaky ingest; broken train)
#       ingest -> report                    (independent branch)
demo = Pipeline("demo")
attempts = {"count": 0}

@demo.task("ingest", retries=2)
def ingest(ctx, dep):
    attempts["count"] += 1
    if attempts["count"] < 3:
        raise ConnectionError("source flaked (transient)")   # heals on retry
    return {"rows": 1000}

@demo.task("features", deps=("ingest",))
def features(ctx, dep):
    return {"n_features": 12}

@demo.task("train", deps=("features",))
def train(ctx, dep):
    raise ValueError("bad hyperparameter file (permanent)")  # never heals

@demo.task("report", deps=("ingest",))
def report(ctx, dep):
    return "daily data report"

states = demo.run("2026-08-30")
print(states)
assert states["ingest"] == "SUCCESS" and attempts["count"] == 3, \
    "transient failure absorbed by retries"
assert states["train"] == "FAILED"
assert states["report"] == "SUCCESS", \
    "one branch's failure did not take down independent work"

# --- Backfill: re-running past days re-computes only what's missing.
backfill = Pipeline("backfill")
executed = []
@backfill.task("etl")
def etl(ctx, dep):
    executed.append(ctx["day"])
    return f"partition-{ctx['day']}"

for day in ("d1", "d2", "d3"):
    backfill.run(day, {"day": day})
executed.clear()
for day in ("d1", "d2", "d3", "d4", "d5"):        # backfill request: 5 days
    backfill.run(day, {"day": day})
print(f"backfill executed only: {executed}")
assert executed == ["d4", "d5"], "cached days skipped - backfills cost only the gap"

Everything Airflow-shaped is here in miniature: the DAG *is* the documentation of your
pipeline, retries absorb the transient, permanent failures stop exactly their blast
radius (nb 00's fault containment, again), and idempotent tasks keyed by run context make
reruns and backfills free. The missing production pieces — distributed workers, sensors,
resource queues, UIs — are engineering, not concept.

## 2. Build: a model registry

The registry answers, at 3 a.m., *"what exactly is in production, and what do we roll
back to?"* It is nb 17's experiment ledger with **lifecycle** and **enforcement**: models
move `registered → staging → production → archived`, and the door to production has
machine-checked preconditions (an evaluation report, passing behavioral tests, and
beating the incumbent) — a gate, not a convention:

In [ ]:
class PromotionError(Exception):
    pass

class ModelRegistry:
    def __init__(self):
        self.versions: list[dict] = []
        self.prod_history: list[int] = []        # stack of production versions

    def register(self, artifact: bytes, metrics: dict, data_version: str,
                 feature_version: str) -> int:
        version = len(self.versions) + 1
        self.versions.append({
            "version": version,
            "artifact_hash": hashlib.blake2b(artifact, digest_size=8).hexdigest(),
            "metrics": metrics,
            "data_version": data_version,
            "feature_version": feature_version,   # nb 16: model+features = one unit
            "stage": "registered",
        })
        return version

    def get(self, version: int) -> dict:
        return self.versions[version - 1]

    def production(self) -> dict | None:
        return self.get(self.prod_history[-1]) if self.prod_history else None

    def promote(self, version: int) -> None:
        """THE GATE. Every condition here is a lesson from an earlier notebook."""
        record = self.get(version)
        m = record["metrics"]
        if "holdout_auc" not in m:
            raise PromotionError("no evaluation report (nb 17)")
        if not m.get("behavioral_tests_pass"):
            raise PromotionError("behavioral tests failing (nb 17 §4.3)")
        if m.get("min_slice_auc", 0) < 0.5:
            raise PromotionError("a slice is worse than a coin flip (nb 17 §4.2)")
        incumbent = self.production()
        if incumbent and m["holdout_auc"] <= incumbent["metrics"]["holdout_auc"]:
            raise PromotionError(
                f"does not beat incumbent ({m['holdout_auc']:.3f} <= "
                f"{incumbent['metrics']['holdout_auc']:.3f})")
        if incumbent:
            incumbent["stage"] = "archived"
        record["stage"] = "production"
        self.prod_history.append(version)

    def rollback(self) -> int:
        """One command, because everything is versioned and immutable (nb 18)."""
        if len(self.prod_history) < 2:
            raise PromotionError("nothing to roll back to")
        retired = self.prod_history.pop()
        self.get(retired)["stage"] = "archived"
        current = self.prod_history[-1]
        self.get(current)["stage"] = "production"
        return current

registry = ModelRegistry()
print("registry ready: stages + gates + one-command rollback")

## 3. The loop, assembled

Now compose the course. One pipeline runs the whole continual-learning cycle from
nb 14–20, with the registry gate deciding promotion. We run it twice: once with a healthy
retrain (promotes), once with a degraded candidate — a broken upstream sample, nb 19's
"pipeline failure" impostor — which the gate must stop *without human eyes*:

In [ ]:
def make_world(n, seed, broken=False):
    r = np.random.default_rng(seed)
    X = r.normal(0, 1, (n, 4))
    y = ((X[:, 0] + 0.7 * X[:, 1] - 0.4 * X[:, 2] + r.normal(0, 0.7, n)) > 0).astype(int)
    if broken:
        X = X.copy()
        X[:, :2] = 0.0            # an upstream job nulled two feature columns
    return X, y

X_holdout, y_holdout = make_world(4_000, seed=999)
mlops = Pipeline("continual-learning")

@mlops.task("ingest", retries=1)
def t_ingest(ctx, dep):
    X, y = make_world(6_000, seed=ctx["seed"], broken=ctx.get("broken", False))
    return {"X": X, "y": y, "data_version": f"lake@{ctx['run_day']}"}

@mlops.task("validate", deps=("ingest",))
def t_validate(ctx, dep):
    X = dep["ingest"]["X"]
    zero_share = float((X == 0).mean())
    if zero_share > 0.4:          # nb 14's audit: fail loud, fail EARLY...
        raise ValueError(f"suspicious zeros: {zero_share:.0%} of feature values")
    return {"rows": len(X), "zero_share": zero_share}

@mlops.task("train", deps=("ingest", "validate"))
def t_train(ctx, dep):
    X, y = dep["ingest"]["X"], dep["ingest"]["y"]
    return {"model": LogisticRegression().fit(X, y)}

@mlops.task("evaluate", deps=("train",))
def t_evaluate(ctx, dep):
    from sklearn.metrics import roc_auc_score
    model = dep["train"]["model"]
    p = model.predict_proba(X_holdout)[:, 1]
    slice_mask = X_holdout[:, 3] > 1      # an arbitrary monitored segment
    return {"holdout_auc": float(roc_auc_score(y_holdout, p)),
            "min_slice_auc": float(roc_auc_score(y_holdout[slice_mask], p[slice_mask])),
            "behavioral_tests_pass": bool(np.isfinite(p).all() and (0 <= p).all())}

@mlops.task("promote", deps=("train", "evaluate", "ingest"))
def t_promote(ctx, dep):
    version = registry.register(
        artifact=json.dumps(dep["train"]["model"].coef_.tolist()).encode(),
        metrics=dep["evaluate"],
        data_version=dep["ingest"]["data_version"],
        feature_version="features@v3",
    )
    registry.promote(version)             # raises PromotionError if unworthy
    return {"promoted": version}

# ---- Run 1: a healthy retrain.
print("run 1 (healthy):")
s1 = mlops.run("day-100", {"run_day": "day-100", "seed": 100})
prod_after_1 = registry.production()["version"]
print(f"  states: {s1}\n  production: v{prod_after_1}")
assert s1["promote"] == "SUCCESS" and prod_after_1 == 1

# ---- Run 2: upstream corruption. The pipeline must defend itself.
print("run 2 (broken upstream):")
s2 = mlops.run("day-101", {"run_day": "day-101", "seed": 101, "broken": True})
print(f"  states: {s2}\n  production: v{registry.production()['version']}")
assert s2["validate"] == "FAILED", "the data audit caught it before any training"
assert s2["train"] == s2["promote"] == "UPSTREAM_FAILED"
assert registry.production()["version"] == 1, "production untouched by the bad run"

# ---- Run 3: data OK but the model is mediocre (small corrupted-ish sample).
print("run 3 (trains, but worse than incumbent):")
s3 = mlops.run("day-102", {"run_day": "day-102", "seed": 5, "broken": False})
# Force a weak candidate by evaluating a model trained on 60 rows:
weak = LogisticRegression().fit(*make_world(60, seed=7))
from sklearn.metrics import roc_auc_score
weak_metrics = {
    "holdout_auc": float(roc_auc_score(y_holdout, weak.predict_proba(X_holdout)[:, 1])),
    "min_slice_auc": 0.55, "behavioral_tests_pass": True,
}
v_weak = registry.register(b"weak", weak_metrics, "lake@day-102", "features@v3")
try:
    registry.promote(v_weak)
    raise AssertionError("the gate must hold")
except PromotionError as e:
    print(f"  gate held: {e}")
assert registry.production()["version"] in (1, 2), "prod is whatever last EARNED it"

# ---- And the 3 a.m. move:
if len(registry.prod_history) >= 2:
    restored = registry.rollback()
    print(f"  rollback drill: production now v{restored}")
print("the loop runs, defends itself, and reverses on command")

The shape to internalize: **every check a human once made became a task or a gate.**
nb 14's audit is `validate`; nb 17's evaluation discipline is `evaluate` + the gate's
preconditions; nb 20's champion/challenger is the `holdout_auc` comparison; nb 18's
rollback is a registry pop. The pipeline is the institutional memory of every incident
that ever taught you a check.

## 4. Build vs buy, and the humans

**Build vs buy.** The layers sort cleanly: storage/compute — always buy (cloud).
Orchestrator, tracker, registry — adopt open source; you now know exactly what they do
inside, which is what this notebook was for (build-to-understand, buy-to-run).
Feature store — adopt if your org has the nb 16 skew problem across several teams;
overkill for one model. Serving — start with the simplest thing meeting nb 18's numbers.
The rule: **buy commodity capabilities, build only what differentiates you** — and your
drift monitors' thresholds, your promotion gates, your label pipelines are usually the
differentiators, because they encode *your* domain's failure history.

**Team shape** (DMLS ch. 11's closing argument): the alternative to a platform is not
"no platform" — it's every scientist hand-rolling brittle halves of one. Mature orgs
converge on a **platform team** (owns the machinery above) enabling **end-to-end
owners** (a scientist/engineer who takes a problem from framing (nb 13) to monitoring
(nb 19) — possible *only because* the platform absorbed the undifferentiated toil).
Success metric for the platform: how little its users must think about it — the same
abstraction test as nb 00's maintainability.

## 5. Exercises

1. **Cycles.** Our `_toposort` refuses cycles at run time. Why is "detect at DAG
   registration" better, and what legitimate-looking workflow pattern (retraining on
   its own predictions?) is actually a *disguised* cycle that no topological check can
   catch (hint: nb 19 §4)?
2. **Idempotency audit.** Which of our pipeline's tasks are safely re-runnable as
   written, and what one change makes `promote` idempotent (so a crash between
   `register` and `promote`, then a retry, doesn't double-register)? (nb 11 §2 supplies
   the pattern and the vocabulary.)
3. **Gate design.** Add three more preconditions to `ModelRegistry.promote` drawn from
   nbs 16–19, each with the incident it prevents.
4. **Startup triage.** A 15-person company with one ML product: rank these by "adopt
   now / adopt at 5 models / build never": experiment tracker, DAG orchestrator, feature
   store, model registry, drift monitoring, GPU cluster manager.
5. **The 3 a.m. runbook.** Using this notebook's objects, write the exact sequence of
   calls (and checks) for: prod model misbehaving → confirm → rollback → open incident →
   schedule retrain. Which single missing piece of metadata would make step 1
   impossible?

### Solutions

**1.** Registration-time checks fail at *deploy* (cheap, in CI) instead of at 2 a.m.
mid-run with half the DAG executed. The undetectable cycle: the *data* cycle — model v2
trains on data generated under model v1's decisions (recommendations, approvals). The
DAG of tasks is acyclic; the DAG of *influence* is not — nb 19's feedback loop, wearing
an orchestrator costume. No topological check sees it; only propensity logging and
exploration budgets do.

**2.** `ingest`/`train`/`evaluate` are pure functions of (ctx, deps) — re-runnable.
`validate` too. `promote` mutates the registry: a retry after partial completion could
register twice. Fix: make registration idempotent via a natural key — register keyed on
`(data_version, feature_version, artifact_hash)` and return the existing version on
repeat (an idempotency key, nb 11 §2, applied to a control-plane write).

**3.** (a) *Feature-version compatibility*: refuse if `feature_version` isn't deployed
in serving — prevents nb 16's train/serve skew shipping as an incident. (b) *Data
freshness*: refuse if `data_version` is older than N days — prevents nb 19's "retrained
on pre-drift data" non-fix (our nb 20 §1 lesson). (c) *Latency/size budget*: refuse if
the artifact exceeds nb 18's serving envelope — prevents the model that wins offline
and times out online. (Bonus: calibration ECE bound, nb 17 — protects every nb 13
threshold downstream.)

**4.** Adopt now: experiment tracker (costs nothing, un-losable results) and a basic
registry (even a conventions-only one — you cannot ship safely without knowing what's
where). Adopt at ~5 models: orchestrator (before that, cron+scripts with idempotent
tasks survive), drift monitoring as shared tooling (the *checks* you need from day one,
the *platform* later). Feature store: only when team #2 re-implements team #1's
features. GPU cluster manager: build never, buy/rent when deep learning actually
arrives.

**5.** (1) Confirm: `registry.production()` → version + `data_version` +
`feature_version` + metrics — compare live telemetry (nb 19 panel) against the
registered evaluation. (2) `registry.rollback()` — one call *because* prod_history is a
stack of immutable versions. (3) Incident log links the run_key from
`pipeline.history`. (4) Schedule retrain: `mlops.run(new_day, ctx)` once the data is
vetted. The metadata that makes step 1 possible at all: the **artifact hash ↔ serving
deployment** linkage — if serving can't tell you which registered version it is
actually running, every other record is decoration.

## 6. Takeaways — and Part 2, folded up

- Orchestration = DAG + retries + isolation + idempotent, context-keyed tasks (which
  make backfills free). The DAG is executable documentation; cron is not a dependency
  model.
- The registry turns "which model is live?" into a data-plane fact with **enforced
  gates** — every gate a fossilized incident. Rollback must be a pop, not a project.
- Build to understand; buy/adopt to run; own only your differentiators (gates,
  monitors, label pipelines — your encoded scar tissue).
- **Part 2 in one loop**: frame with economics (13) → feed from the lake and the log
  (14) → engineer examples honestly (15) → features without leaks or skew (16) →
  evaluate past the average (17) → serve within physics (18) → monitor for the world's
  betrayal (19) → retrain and prove it safely (20) → automate all of it (21). Every
  stage stood on Part 1's data-systems bedrock.

**Further reading.** DMLS ch. 10–11; Sculley et al., *Hidden Technical Debt in ML
Systems* (2015) — reread it now that you've built the debts; the Airflow and MLflow
architecture docs (compare with what you built today).

**Next:** Part 3 — the capstones. [22 · URL shortener](../part3-capstones/22-capstone-url-shortener.ipynb)
begins: everything from Part 1, one system at a time, end to end.